In [0]:
%pip install statsmodels arch

In [0]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

df = (spark.table("workspace.vol_project.gold_btc_daily_rv")
      .select("date", "rv", "log_rv", "has_gap")
      .toPandas())
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

In [0]:
W, M = 7, 30   # weekly and monthly windows (your decision)

df["x_d"] = np.log(df["rv"])                      # today's RV
df["x_w"] = np.log(df["rv"].rolling(W).mean())    # avg RV, last 7 days
df["x_m"] = np.log(df["rv"].rolling(M).mean())    # avg RV, last 30 days
df["y"]   = df["log_rv"].shift(-1)                # TARGET: tomorrow's log RV

data = df.dropna()
print(len(data))

In [0]:
X = sm.add_constant(data[["x_d", "x_w", "x_m"]])
full = sm.OLS(data["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 30})
print(full.summary())

In [0]:
features = ["x_d", "x_w", "x_m"]
test_idx = data.index[data.index >= "2024-01-01"]

def har_walk_forward(data, window=None):
    preds = {}
    for t in test_idx:
        train = data[data.index < t]
        if window:
            train = train.iloc[-window:]
        model = sm.OLS(train["y"], sm.add_constant(train[features])).fit()
        x_t = sm.add_constant(data.loc[[t], features], has_constant="add")
        log_pred = model.predict(x_t).iloc[0]
        preds[t] = np.exp(log_pred + 0.5 * model.resid.var())
    return pd.Series(preds)

har_expanding = har_walk_forward(data)               # all history
har_rolling   = har_walk_forward(data, window=730)   # last 2 years only

In [0]:
actual = np.exp(data.loc[test_idx, "y"])
naive  = data.loc[test_idx, "rv"]          # tomorrow = today

def qlike(a, f):
    r = a / f
    return (r - np.log(r) - 1).mean()

def mse(a, f):
    return ((a - f) ** 2).mean()

results = pd.DataFrame({
    "QLIKE": [qlike(actual, f) for f in [naive, har_expanding, har_rolling]],
    "MSE":   [mse(actual, f)   for f in [naive, har_expanding, har_rolling]],
}, index=["Naive", "HAR expanding", "HAR rolling (730d)"])

results["QLIKE vs naive"] = results["QLIKE"] / results.loc["Naive", "QLIKE"]
results["MSE vs naive"]   = results["MSE"]   / results.loc["Naive", "MSE"]
results

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.vol_project.forecasts (
    target_date DATE,
    model       STRING,
    forecast    DOUBLE,
    actual      DOUBLE,
    created_at  TIMESTAMP
)
""")

In [0]:
from pyspark.sql import functions as F

fc = pd.concat([
    pd.DataFrame({"model": "naive",           "forecast": naive}),
    pd.DataFrame({"model": "har_expanding",   "forecast": har_expanding}),
    pd.DataFrame({"model": "har_rolling_730", "forecast": har_rolling}),
])
fc["actual"] = actual.reindex(fc.index).values
fc.index.name = "origin_date"
fc = fc.reset_index()

# a forecast made at the end of day t is FOR day t+1
fc["target_date"] = (fc["origin_date"] + pd.Timedelta(days=1)).dt.date
fc = fc[["target_date", "model", "forecast", "actual"]]

print(fc.groupby("model").size())

In [0]:
models = fc["model"].unique().tolist()
model_list = ", ".join(f"'{m}'" for m in models)

(spark.createDataFrame(fc)
    .withColumn("created_at", F.current_timestamp())
    .select("target_date", "model", "forecast", "actual", "created_at")
    .write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"model IN ({model_list})")
    .saveAsTable("workspace.vol_project.forecasts"))

In [0]:
models = fc["model"].unique().tolist()
model_list = ", ".join(f"'{m}'" for m in models)

(spark.createDataFrame(fc)
    .withColumn("created_at", F.current_timestamp())
    .select("target_date", "model", "forecast", "actual", "created_at")
    .write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"model IN ({model_list})")
    .saveAsTable("workspace.vol_project.forecasts"))

In [0]:
display(spark.sql("""
SELECT model,
       COUNT(*)          AS n,
       MIN(target_date)  AS first_day,
       MAX(target_date)  AS last_day,
       AVG(actual / forecast - LN(actual / forecast) - 1) AS qlike
FROM workspace.vol_project.forecasts
GROUP BY model
ORDER BY qlike
"""))